# `construction_year` 03: related-feature analysis

            **Purpose:** identify predictors that represent the same concept, form a
            hierarchy, share a missingness process or plausibly interact with
            `construction_year`.

            ## Relationships selected in advance

            - `date_recorded` — Together they define waterpoint age at observation.
- `installer` — Installers operate in particular construction cohorts.
- `extraction_type` — Extraction technology changes across construction cohorts.
- `gps_height` — Year and height zeros frequently share one measurement block.


In [1]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display


def find_stage_directory():
    start = Path.cwd().resolve()
    for candidate in (start, *start.parents):
        if (
            (candidate / "data" / "TrainingSetValues.csv").exists()
            and (candidate / "src" / "source_data_validation.py").exists()
        ):
            return candidate
    raise FileNotFoundError("Could not locate the stage-1-pump-it-up directory.")


stage_directory = find_stage_directory()
source_directory = str((stage_directory / "src").resolve())
if source_directory not in sys.path:
    sys.path.insert(0, source_directory)

from predictor_audit import (
    analysis_categories,
    categorical_summary,
    categorical_target_profile,
    category_frequency_table,
    numeric_summary,
    numeric_target_summary,
    related_feature_summary,
    sentinel_mask,
    source_blank_mask,
    text_normalisation_summary,
)
from source_data_validation import (
    validate_aligned_ids,
    validate_label_frame,
    validate_raw_feature_schema,
)

data_directory = stage_directory / "data"
training_features = pd.read_csv(
    data_directory / "TrainingSetValues.csv",
    keep_default_na=False,
)
training_labels = pd.read_csv(
    data_directory / "TrainingSetLabels.csv",
    keep_default_na=False,
)
test_features = pd.read_csv(
    data_directory / "TestSetValues.csv",
    keep_default_na=False,
)

validate_raw_feature_schema(training_features)
validate_raw_feature_schema(test_features)
validate_label_frame(training_labels)
validate_aligned_ids(training_features, training_labels)

training_data = training_features.merge(
    training_labels,
    on="id",
    validate="one_to_one",
)

feature = 'construction_year'
feature_metadata = {'order': 23, 'name': 'construction_year', 'audit_type': 'year', 'role': 'candidate', 'disposition': 'derive valid pump age and retain unknown-year state', 'finding': 'Year zero affects about a third of rows and a small number of derived ages are negative.', 'decision': 'Keep unknown and inconsistent flags plus valid pump age or construction cohort.', 'risk': 'Age is missing for many rows and is confounded with technology and geography.', 'sentinel_values': [0], 'related': [{'feature': 'date_recorded', 'reason': 'Together they define waterpoint age at observation.'}, {'feature': 'installer', 'reason': 'Installers operate in particular construction cohorts.'}, {'feature': 'extraction_type', 'reason': 'Extraction technology changes across construction cohorts.'}, {'feature': 'gps_height', 'reason': 'Year and height zeros frequently share one measurement block.'}]}
feature_types = {'amount_tsh': 'numeric', 'date_recorded': 'date', 'funder': 'high-cardinality-category', 'gps_height': 'numeric', 'installer': 'high-cardinality-category', 'longitude': 'coordinate', 'latitude': 'coordinate', 'wpt_name': 'high-cardinality-category', 'num_private': 'numeric', 'basin': 'category', 'subvillage': 'high-cardinality-category', 'region': 'category', 'region_code': 'category', 'district_code': 'category', 'lga': 'category', 'ward': 'high-cardinality-category', 'population': 'numeric', 'public_meeting': 'binary', 'recorded_by': 'constant', 'scheme_management': 'category', 'scheme_name': 'high-cardinality-category', 'permit': 'binary', 'construction_year': 'year', 'extraction_type': 'category', 'extraction_type_group': 'category', 'extraction_type_class': 'category', 'management': 'category', 'management_group': 'category', 'payment': 'category', 'payment_type': 'category', 'water_quality': 'category', 'quality_group': 'category', 'quantity': 'category', 'quantity_group': 'category', 'source': 'category', 'source_type': 'category', 'source_class': 'category', 'waterpoint_type': 'category', 'waterpoint_type_group': 'category'}
assert feature in training_features.columns
print(
    f"Validated {len(training_features):,} training rows and "
    f"{len(test_features):,} test rows for {feature}."
)


Validated 59,400 training rows and 14,850 test rows for construction_year.


In [2]:
relationship_inventory = pd.DataFrame(feature_metadata["related"])
display(relationship_inventory)

relationship_evidence = related_feature_summary(
    training_features,
    feature,
    feature_metadata["audit_type"],
    feature_metadata["related"],
    feature_types,
)
display(relationship_evidence)


,feature,reason
0,date_recorded,Together they define waterpoint age at observa...
1,installer,Installers operate in particular construction ...
2,extraction_type,Extraction technology changes across construct...
3,gps_height,Year and height zeros frequently share one mea...


,primary,related,measure,association,complete rows,primary levels,related levels,forward modal purity (%),reverse modal purity (%),relationship rationale
0,construction_year,date_recorded,Spearman correlation,0.1877,59400,55,356,NaN,NaN,Together they define waterpoint age at observa...
1,construction_year,installer,correlation ratio (eta),0.6311,59400,55,2146,NaN,NaN,Installers operate in particular construction ...
2,construction_year,extraction_type,correlation ratio (eta),0.3189,59400,55,18,NaN,NaN,Extraction technology changes across construct...
3,construction_year,gps_height,Spearman correlation,0.6123,59400,55,2428,NaN,NaN,Year and height zeros frequently share one mea...


In [3]:
recording_year = pd.to_datetime(
    training_features["date_recorded"],
    errors="coerce",
).dt.year
construction_year = pd.to_numeric(
    training_features["construction_year"],
    errors="coerce",
)
age = recording_year.sub(construction_year).where(construction_year.gt(0))
age_check = pd.DataFrame({
    "known construction years": [construction_year.gt(0).sum()],
    "unknown year rows": [construction_year.eq(0).sum()],
    "negative derived ages": [age.lt(0).sum()],
    "median non-negative age": [age.where(age.ge(0)).median()],
    "maximum non-negative age": [age.where(age.ge(0)).max()],
}, index=["date_recorded - construction_year"])
display(age_check)


,known construction years,unknown year rows,negative derived ages,median non-negative age,maximum non-negative age
date_recorded - construction_year,38691,20709,9,13.0,53.0


## Discussion and modelling consequence

The relationships above were nominated before inspecting the pairwise
coefficients. A strong association can mean useful interaction, hierarchy,
shared collection behaviour or redundancy; it is not a reason to keep both
fields automatically.

For `construction_year`, carry the relationships into controlled ablations
and fit every learned grouping or encoding inside the training fold. The
current provisional disposition remains: **derive valid pump age and retain unknown-year state**.
